# DATA 612 - Project 1
## Global Baseline Predictors and RMSE

### K-Drama Recommendation System

Chhiring Lama

DATA 612

May 2026

## Business Perspective

This project builds a simple K-drama recommendation system. The system predicts user ratings for different K-dramas based on historical ratings data. By identifying patterns in user preferences and drama popularity, the system can estimate ratings and support personalized recommendations. Similar recommendation systems are commonly used by streaming platforms to help users discover content they may enjoy.

## Dataset Description

A toy dataset was created for this project using K-drama ratings. The dataset contains seven users (Karma, Will, David, Kent, Adren, Lucy, and Sean) and seven K-dramas (Move to Heaven, Flower of Evil, Hospital Playlist, Reply 1988, Extraordinary Attorney Woo, Alchemy of Souls, and Weak Hero Class 1). Ratings were assigned on a 1–5 scale, where 1 represents the lowest rating and 5 represents the highest rating. Several ratings were intentionally left missing to simulate real-world recommendation scenarios.

Because the dataset was manually created for educational purposes, no additional data cleaning was required.

In [16]:
import pandas as pd
import numpy as np

ratings = pd.DataFrame({
    "User": ["Karma", "Will", "David", "Kent", "Adren", "Lucy", "Sean"],
    "Move to Heaven": [5, 4, 3, 5, 2, 4, np.nan],
    "Flower of Evil": [4, 5, 4, np.nan, 2, 5, 1],
    "Hospital Playlist": [np.nan, 4, 5, 4, 3, np.nan, 2],
    "Reply 1988": [5, np.nan, 4, 5, np.nan, 4, 2],
    "Extraordinary Attorney Woo": [4, 5, np.nan, 4, 2, 5, 1],
    "Alchemy of Souls": [5, 4, 4, 5, 3, 4, 2],
    "Weak Hero Class 1": [4, 5, 4, 5, 2, 5, 1]
})

ratings

,User,Move to Heaven,Flower of Evil,Hospital Playlist,Reply 1988,Extraordinary Attorney Woo,Alchemy of Souls,Weak Hero Class 1
0,Karma,5.0,4.0,NaN,5.0,4.0,5,4
1,Will,4.0,5.0,4.0,NaN,5.0,4,5
2,David,3.0,4.0,5.0,4.0,NaN,4,4
3,Kent,5.0,NaN,4.0,5.0,4.0,5,5
4,Adren,2.0,2.0,3.0,NaN,2.0,3,2
5,Lucy,4.0,5.0,NaN,4.0,5.0,4,5
6,Sean,NaN,1.0,2.0,2.0,1.0,2,1


## User-Item Matrix

The ratings data was organized into a user-item matrix where each row represents a user and each column represents a K-drama. The matrix contains user ratings on a 1–5 scale, and missing values are represented as NaN. This matrix serves as the foundation for calculating rating predictions, user bias, item bias, and RMSE.

In [17]:
ratings

,User,Move to Heaven,Flower of Evil,Hospital Playlist,Reply 1988,Extraordinary Attorney Woo,Alchemy of Souls,Weak Hero Class 1
0,Karma,5.0,4.0,NaN,5.0,4.0,5,4
1,Will,4.0,5.0,4.0,NaN,5.0,4,5
2,David,3.0,4.0,5.0,4.0,NaN,4,4
3,Kent,5.0,NaN,4.0,5.0,4.0,5,5
4,Adren,2.0,2.0,3.0,NaN,2.0,3,2
5,Lucy,4.0,5.0,NaN,4.0,5.0,4,5
6,Sean,NaN,1.0,2.0,2.0,1.0,2,1


In [18]:
ratings.shape

(7, 8)

In [19]:
ratings_matrix = ratings.set_index("User")

ratings_matrix

,Move to Heaven,Flower of Evil,Hospital Playlist,Reply 1988,Extraordinary Attorney Woo,Alchemy of Souls,Weak Hero Class 1
User,,,,,,,
Karma,5.0,4.0,NaN,5.0,4.0,5,4
Will,4.0,5.0,4.0,NaN,5.0,4,5
David,3.0,4.0,5.0,4.0,NaN,4,4
Kent,5.0,NaN,4.0,5.0,4.0,5,5
Adren,2.0,2.0,3.0,NaN,2.0,3,2
Lucy,4.0,5.0,NaN,4.0,5.0,4,5
Sean,NaN,1.0,2.0,2.0,1.0,2,1


In [20]:
ratings_matrix.shape


(7, 7)

In [21]:
# Create a copy of the original user-item matrix.
# The copy will be used as the training dataset so that
# the original ratings matrix remains unchanged.
train_matrix = ratings_matrix.copy()

train_matrix

,Move to Heaven,Flower of Evil,Hospital Playlist,Reply 1988,Extraordinary Attorney Woo,Alchemy of Souls,Weak Hero Class 1
User,,,,,,,
Karma,5.0,4.0,NaN,5.0,4.0,5,4
Will,4.0,5.0,4.0,NaN,5.0,4,5
David,3.0,4.0,5.0,4.0,NaN,4,4
Kent,5.0,NaN,4.0,5.0,4.0,5,5
Adren,2.0,2.0,3.0,NaN,2.0,3,2
Lucy,4.0,5.0,NaN,4.0,5.0,4,5
Sean,NaN,1.0,2.0,2.0,1.0,2,1


## Train-Test Split

To evaluate the predictors, selected known ratings were removed from the training matrix and stored separately as test ratings. The model is built using the remaining training ratings and evaluated by comparing predictions against the hidden test ratings.

In [22]:
# Create a test set by hiding selected ratings.
# These ratings will be used later to evaluate prediction accuracy.

test_ratings = {
    ("Karma", "Flower of Evil"): 4.0,
    ("Will", "Weak Hero Class 1"): 5.0,
    ("David", "Reply 1988"): 4.0,
    ("Kent", "Move to Heaven"): 5.0,
    ("Lucy", "Extraordinary Attorney Woo"): 5.0
}

for (user, drama), rating in test_ratings.items():
    train_matrix.loc[user, drama] = np.nan

train_matrix

,Move to Heaven,Flower of Evil,Hospital Playlist,Reply 1988,Extraordinary Attorney Woo,Alchemy of Souls,Weak Hero Class 1
User,,,,,,,
Karma,5.0,NaN,NaN,5.0,4.0,5,4.0
Will,4.0,5.0,4.0,NaN,5.0,4,NaN
David,3.0,4.0,5.0,NaN,NaN,4,4.0
Kent,NaN,NaN,4.0,5.0,4.0,5,5.0
Adren,2.0,2.0,3.0,NaN,2.0,3,2.0
Lucy,4.0,5.0,NaN,4.0,NaN,4,5.0
Sean,NaN,1.0,2.0,2.0,1.0,2,1.0


## Raw Average Predictor

The raw average predictor uses the overall average rating from the training data as the predicted rating for every user-item combination. This method does not consider individual user preferences or item popularity. It serves as a simple baseline model for comparison.

In [23]:
# Calculate the global average rating using only the training data.
# Pandas automatically ignores NaN values when calculating the mean.

global_mean = train_matrix.stack().mean()

global_mean

3.5945945945945947

## Raw Average Predictions

Using the global average rating from the training dataset, each hidden test rating is predicted with the same value. These predictions will be compared against the actual ratings to evaluate the performance of the raw average predictor.

In [24]:
# Create raw average predictions for all test ratings

raw_predictions = []

for (user, drama), actual_rating in test_ratings.items():
    raw_predictions.append([
        user,
        drama,
        actual_rating,
        global_mean
    ])

raw_predictions_df = pd.DataFrame(
    raw_predictions,
    columns=["User", "Drama", "Actual Rating", "Predicted Rating"]
)

raw_predictions_df

,User,Drama,Actual Rating,Predicted Rating
0,Karma,Flower of Evil,4.0,3.594595
1,Will,Weak Hero Class 1,5.0,3.594595
2,David,Reply 1988,4.0,3.594595
3,Kent,Move to Heaven,5.0,3.594595
4,Lucy,Extraordinary Attorney Woo,5.0,3.594595


## RMSE for Raw Average Predictor

Root Mean Squared Error (RMSE) measures the average prediction error between the actual ratings and the predicted ratings. Lower RMSE values indicate better prediction accuracy.

In [25]:
# Calculate RMSE for the raw average predictor

from sklearn.metrics import mean_squared_error
import numpy as np

rmse_raw = np.sqrt(
    mean_squared_error(
        raw_predictions_df["Actual Rating"],
        raw_predictions_df["Predicted Rating"]
    )
)

rmse_raw

1.118409598143009

## User Bias

User bias measures whether a user tends to rate items higher or lower than the overall average rating. A positive user bias means the user is generally generous with ratings, while a negative user bias means the user is generally stricter.

In [26]:
# Calculate each user's average rating from the training matrix
# Then subtract the global mean to get user bias

user_means = train_matrix.mean(axis=1)
user_bias = user_means - global_mean

user_bias_df = pd.DataFrame({
    "User Average": user_means,
    "User Bias": user_bias
})

user_bias_df

,User Average,User Bias
User,,
Karma,4.600000,1.005405
Will,4.400000,0.805405
David,4.000000,0.405405
Kent,4.600000,1.005405
Adren,2.333333,-1.261261
Lucy,4.400000,0.805405
Sean,1.500000,-2.094595


## Item Bias

Item bias measures whether a K-drama tends to receive higher or lower ratings than the overall average rating. A positive item bias means the drama is generally rated above average, while a negative item bias means it is generally rated below average.

In [27]:
# Calculate each drama's average rating from the training matrix
# Then subtract the global mean to get item bias

item_means = train_matrix.mean(axis=0)
item_bias = item_means - global_mean

item_bias_df = pd.DataFrame({
    "Drama Average": item_means,
    "Item Bias": item_bias
})

item_bias_df

,Drama Average,Item Bias
Move to Heaven,3.600000,0.005405
Flower of Evil,3.400000,-0.194595
Hospital Playlist,3.600000,0.005405
Reply 1988,4.000000,0.405405
Extraordinary Attorney Woo,3.200000,-0.394595
Alchemy of Souls,3.857143,0.262548
Weak Hero Class 1,3.500000,-0.094595


## Baseline Predictor

The baseline predictor improves upon the raw average predictor by incorporating both user bias and item bias. Predictions are calculated as the sum of the global average rating, the user bias, and the item bias.

Prediction = Global Average + User Bias + Item Bias

In [28]:
# Generate baseline predictions for the hidden test ratings

baseline_predictions = []

for (user, drama), actual_rating in test_ratings.items():

    prediction = (
        global_mean
        + user_bias[user]
        + item_bias[drama]
    )

    baseline_predictions.append([
        user,
        drama,
        actual_rating,
        prediction
    ])

baseline_predictions_df = pd.DataFrame(
    baseline_predictions,
    columns=["User", "Drama", "Actual Rating", "Predicted Rating"]
)

baseline_predictions_df

,User,Drama,Actual Rating,Predicted Rating
0,Karma,Flower of Evil,4.0,4.405405
1,Will,Weak Hero Class 1,5.0,4.305405
2,David,Reply 1988,4.0,4.405405
3,Kent,Move to Heaven,5.0,4.605405
4,Lucy,Extraordinary Attorney Woo,5.0,4.005405


## RMSE for Baseline Predictor

The baseline predictor incorporates user and item biases to generate personalized rating predictions. RMSE is used to compare its performance against the raw average predictor.

In [29]:
# Calculate RMSE for the baseline predictor

rmse_baseline = np.sqrt(
    mean_squared_error(
        baseline_predictions_df["Actual Rating"],
        baseline_predictions_df["Predicted Rating"]
    )
)

rmse_baseline

0.6254745459223627

## Training and Test RMSE Comparison

The models were evaluated on both the training data and the hidden test ratings. Training RMSE measures how well the model fits the ratings used to build the predictor, while test RMSE measures how well the model predicts ratings that were hidden from the model.

In [30]:
# Calculate training RMSE for the raw average predictor

train_actual = train_matrix.stack()

train_raw_predictions = pd.Series(
    global_mean,
    index=train_actual.index
)

rmse_raw_train = np.sqrt(
    mean_squared_error(train_actual, train_raw_predictions)
)

# Calculate training RMSE for the baseline predictor

train_baseline_predictions = []

for user, drama in train_actual.index:
    prediction = global_mean + user_bias[user] + item_bias[drama]
    train_baseline_predictions.append(prediction)

rmse_baseline_train = np.sqrt(
    mean_squared_error(train_actual, train_baseline_predictions)
)

# Create final RMSE comparison table

final_results = pd.DataFrame({
    "Model": ["Raw Average Predictor", "Baseline Predictor"],
    "Train RMSE": [rmse_raw_train, rmse_baseline_train],
    "Test RMSE": [rmse_raw, rmse_baseline]
})

final_results

,Model,Train RMSE,Test RMSE
0,Raw Average Predictor,1.304037,1.118410
1,Baseline Predictor,0.503728,0.625475


## Conclusion

This project developed a simple K-drama recommendation system using a user-item rating matrix. Two prediction approaches were evaluated: a raw average predictor and a baseline predictor incorporating user and item biases.

The raw average predictor produced a training RMSE of approximately 1.30 and a test RMSE of approximately 1.12. The baseline predictor produced a training RMSE of approximately 0.50 and a test RMSE of approximately 0.63.

Since the baseline predictor resulted in lower RMSE values for both the training and test datasets, it provided more accurate rating predictions than the raw average predictor. The results demonstrate that accounting for user preferences and item popularity improves recommendation accuracy compared to relying only on the overall average rating.